In [98]:
import json
import requests
import copy
import pandas as pd
from pathlib import Path
from my_secrets import secrets

Set up basic folder structure

In [99]:
# Base path
curr_dir = Path()

# LDA folders
lda_dir = curr_dir / 'LDA'
lda_input_dir = lda_dir / 'input'
lda_output_dir = lda_dir / 'output'

# Congress folders
congress_dir = curr_dir / 'Congress'
congress_input_dir = congress_dir / 'input'
congress_output_dir = congress_dir / 'output'

## LDA

In [100]:
lda_root = "https://lda.senate.gov/api/v1/"

### Overview of root endpoint categories

In [101]:
r = requests.get(lda_root).content
lda_categories = json.loads(r)
for lda_cat, lda_cat_url in lda_categories.items():
    print(f"Requesting {lda_cat} at {lda_cat_url}...")
    try:
        lda_cat_content = requests.get(lda_cat_url).content
        lda_cat_content = json.loads(lda_cat_content)
        print("\t", lda_cat_content)
    except Exception as e:
        print(f"Failed!\n{e}")

Requesting filings at https://lda.senate.gov/api/v1/filings/...
	 {'count': 1948519, 'next': 'https://lda.senate.gov/api/v1/filings/?page=2', 'previous': None, 'results': [{'url': 'https://lda.senate.gov/api/v1/filings/455edc06-55d1-41ed-878e-70a4040f953c/', 'filing_uuid': '455edc06-55d1-41ed-878e-70a4040f953c', 'filing_type': 'MM', 'filing_type_display': 'Mid-Year Report', 'filing_year': 1999, 'filing_period': 'mid_year', 'filing_period_display': 'Mid-Year (Jan 1 - Jun 30)', 'filing_document_url': 'https://lda.senate.gov/filings/public/filing/455edc06-55d1-41ed-878e-70a4040f953c/print/', 'filing_document_content_type': 'application/pdf', 'income': None, 'expenses': None, 'expenses_method': None, 'expenses_method_display': None, 'posted_by_name': None, 'dt_posted': '1905-06-24T00:00:00-05:00', 'termination_date': None, 'registrant_country': 'United States of America', 'registrant_ppb_country': 'United States of America', 'registrant_address_1': None, 'registrant_address_2': None, 'regi

### Filings

#### Generate examples

In [102]:
url = lda_categories['filings']
filings_data = []
limit = 100
for year in range(2026, 2019, -1):

    if len(filings_data) >= limit:
        break

    print(year)
    page = 1
    while len(filings_data) < limit and url is not None:
        print("\t", f"page {page}")

        payload = {
            'filing_year': year,
            'ordering':'-dt_posted'
        }

        filings = requests.get(
            url,
            headers=secrets['LDA']['headers'],
            params=payload
        ).json()

        url = filings['next']

        filings_data += filings['results']

        page += 1
print(f"All {limit} records obtained")

2026
	 page 1
	 page 2
	 page 3
	 page 4
All 100 records obtained


In [103]:
lda_sample_input_path = lda_input_dir / f'lda_filings_sample_{limit}.json'
with open(lda_sample_input_path, mode='w') as f:
    f.write(json.dumps(filings_data, indent=4))

#### Parse Examples

In [104]:
with open(lda_sample_input_path, mode='r') as f:
    filings_sample = json.loads(f.read())

# Creating lists to then turn into reference dataframes; ignoring lobbyists and government_entities for now
registrant_list = []
filing_client_list = []
lobbying_activity_list = []
# Passing by reference means the filings_sample list is updated by these changes,
# so a separate empty list doesn't need to be created


for filing in filings_sample:
    # Create a registrant lookup table
    registrant = filing['registrant']
    registrant_list.append(registrant)
    filing['registrant'] = registrant['id']

    # Create a filing client lookup table; client_id is its own key-value pair
    filing_client = filing['client']
    filing_client_list.append(filing_client)
    filing['client'] = filing_client['id']

    # Create a lobbying_activity lookup table
    lobbying_activities = filing['lobbying_activities']
    if lobbying_activities != []:
        for lobbying_activity in lobbying_activities:
            # Keep only lobbyists IDs as FK reference
            lobbyists = lobbying_activity['lobbyists']
            lobbyist_ids = [lobbyist['lobbyist']['id'] for lobbyist in lobbyists]
            lobbying_activity['lobbyist_ids'] = lobbyist_ids
            del lobbying_activity['lobbyists']

            # Keep only Gov entity IDs as FK reference
            gov_entities = lobbying_activity['government_entities']
            gov_entity_ids = [gov_entity['id'] for gov_entity in gov_entities]
            lobbying_activity['gov_entity_ids'] = gov_entity_ids
            del lobbying_activity['government_entities']

            # Assign FK reference back to filings table
            lobbying_activity['filing'] = filing['filing_uuid']

            # Append to list
            lobbying_activity_list.append(lobbying_activity)
    # No longer need an explicit reference in filing table
    del filing['lobbying_activities']

Save as dataframes and export to csv

In [105]:
# Convert to pandas dataframes
filings_sample = pd.DataFrame(filings_sample)
filing_client_sample = pd.DataFrame(filing_client_list)
lobbying_activity_sample = pd.DataFrame(lobbying_activity_list)
registrant_sample = pd.DataFrame(registrant_list)

# Save to .csv's
filings_sample.to_csv(lda_output_dir / f'lda_filings_sample_{limit}.csv')
filing_client_sample.to_csv(lda_output_dir / f'lda_filing_clients_sample_{limit}.csv')
lobbying_activity_sample.to_csv(lda_output_dir / f'lda_lobbying_activities_sample_{limit}.csv')
registrant_sample.to_csv(lda_output_dir / f'lda_registrant_sample_{limit}.csv')

## Congress

In [106]:
congress_root = 'https://api.congress.gov/v3/'

### Bills

In [107]:
bill_endpoint = congress_root + 'bill'

payload = {
  'format':'json',
  'offset':0,
  'limit':limit,
  'fromDateTime':'2026-01-01T00:00:00Z',
  'api_key':secrets['Congress']['headers']['Authorization']
}

bills = requests.get(
  bill_endpoint,
  params=payload
).json()['bills']

In [108]:
# Export input
congress_bills_sample_path = congress_input_dir / f'congress_bills_sample_{limit}.json'
with open(congress_bills_sample_path, mode='w') as f:
    f.write(json.dumps(bills, indent=4))

In [109]:
# Update 'bills' by reference
for bill in bills:
    bill['latestActionDate'] = bill['latestAction']['actionDate']
    bill['latestActionText'] = bill['latestAction']['text']
    del bill['latestAction']

In [110]:
bills = pd.DataFrame(bills)
bills.head()

,congress,number,originChamber,originChamberCode,title,type,updateDate,updateDateIncludingText,url,latestActionDate,latestActionText
0,108,5,House,H,"Entitled the ""English Plus Resolution"".",HCONRES,2026-03-23,2026-03-23,https://api.congress.gov/v3/bill/108/hconres/5...,2003-02-21,Referred to the Subcommittee on Education Reform.
1,108,32,House,H,Expressing the sense of the House of Represent...,HRES,2026-03-23,2026-03-23,https://api.congress.gov/v3/bill/108/hres/32?f...,2003-03-06,Referred to the Subcommittee on the Constitution.
2,108,85,House,H,Congratulating the Messiah College men's socce...,HRES,2026-03-23,2026-03-23,https://api.congress.gov/v3/bill/108/hres/85?f...,2003-03-10,Referred to the Subcommittee on 21st Century C...
3,108,86,House,H,Recognizing the contributions of historically ...,HRES,2026-03-23,2026-03-23,https://api.congress.gov/v3/bill/108/hres/86?f...,2003-03-10,Referred to the Subcommittee on 21st Century C...
4,108,116,House,H,Expressing the sense of the House of Represent...,HRES,2026-03-23,2026-03-23,https://api.congress.gov/v3/bill/108/hres/116?...,2003-03-17,Referred to the Subcommittee on Employer-Emplo...


In [111]:
# Export output
bills.to_csv(congress_output_dir / f'congress_bills_sample_{limit}.csv')

### Bill Details

In [112]:
# We have a preset number of records, so we don't need to use .append()
bill_details_list = [{}] * limit

for i, bill_detail_url in enumerate(bills.loc[:, 'url']):
    bill_details = requests.get(
        bill_detail_url,
        params={
            'api_key': secrets['Congress']['headers']['Authorization']
        }
    ).json()['bill']

    # Already in the Bills table
    del bill_details['latestAction']


    # Unpack specific dictionaries into individual key-value pairs (i.e., columns later on)
    detail_keys = [
        'actions',
        'committees',
        'subjects',
        'summaries',
        'textVersions',
        'titles'
    ]

    for detail_key in detail_keys:
        # Count 
        new_count_key = f'{detail_key[:-1]}Count'
        bill_details[new_count_key] = bill_details[detail_key]['count']

        # URL
        new_url_key = f'{detail_key[:-1]}URL'
        bill_details[new_url_key] = bill_details[detail_key]['url']

        # Delete unpacked dict
        del bill_details[detail_key]

    # Policy area seems to just hold name
    # print(bill_details['policyArea']) # test it out
    bill_details['policyAreaName'] = bill_details['policyArea']['name']
    del bill_details['policyArea']

    # Only need the FK reference to the Member table
    bill_details['sponsorIds'] = [sponsor['bioguideId'] for sponsor in bill_details['sponsors']]
    del bill_details['sponsors']

    bill_details_list[i] = bill_details

In [113]:
bill_details = pd.DataFrame(bill_details_list)
bill_details.head()

,congress,introducedDate,legislationUrl,number,originChamber,originChamberCode,title,type,updateDate,updateDateIncludingText,...,textVersionURL,titleCount,titleURL,policyAreaName,sponsorIds,cosponsors,relatedBills,amendments,committeeReports,notes
0,108,2003-01-07,https://www.congress.gov/bill/108th-congress/h...,5,House,H,"Entitled the ""English Plus Resolution"".",HCONRES,2026-03-23T12:35:20Z,2026-03-23T12:35:20Z,...,https://api.congress.gov/v3/bill/108/hconres/5...,2,https://api.congress.gov/v3/bill/108/hconres/5...,Government Operations and Politics,[S000248],NaN,NaN,NaN,NaN,NaN
1,108,2003-01-27,https://www.congress.gov/bill/108th-congress/h...,32,House,H,Expressing the sense of the House of Represent...,HRES,2026-03-23T12:41:21Z,2026-03-23T12:41:21Z,...,https://api.congress.gov/v3/bill/108/hres/32/t...,2,https://api.congress.gov/v3/bill/108/hres/32/t...,"Civil Rights and Liberties, Minority Issues",[D000096],"{'count': 62, 'countIncludingWithdrawnCosponso...",NaN,NaN,NaN,NaN
2,108,2003-02-13,https://www.congress.gov/bill/108th-congress/h...,85,House,H,Congratulating the Messiah College men's socce...,HRES,2026-03-23T12:44:53Z,2026-03-23T12:44:53Z,...,https://api.congress.gov/v3/bill/108/hres/85/t...,2,https://api.congress.gov/v3/bill/108/hres/85/t...,Commemorations,[P000585],"{'count': 11, 'countIncludingWithdrawnCosponso...",NaN,NaN,NaN,NaN
3,108,2003-02-13,https://www.congress.gov/bill/108th-congress/h...,86,House,H,Recognizing the contributions of historically ...,HRES,2026-03-23T12:44:53Z,2026-03-23T12:44:53Z,...,https://api.congress.gov/v3/bill/108/hres/86/t...,2,https://api.congress.gov/v3/bill/108/hres/86/t...,Commemorations,[R000575],"{'count': 22, 'countIncludingWithdrawnCosponso...",NaN,NaN,NaN,NaN
4,108,2003-02-27,https://www.congress.gov/bill/108th-congress/h...,116,House,H,Expressing the sense of the House of Represent...,HRES,2026-03-23T12:41:21Z,2026-03-23T12:41:21Z,...,https://api.congress.gov/v3/bill/108/hres/116/...,2,https://api.congress.gov/v3/bill/108/hres/116/...,Commemorations,[R000053],NaN,NaN,NaN,NaN,NaN


In [114]:
bill_details.to_csv(
    congress_output_dir / f'congress_bill_details_sample_{limit}.csv'
)

### Actions

In [115]:
bill_actions_list = [] # We don't know how many total actions for all bills there are
bill_actions_source_list = [[]] * limit # We'll want to save an example of what we used as a source

sourceSystemMapper = {
    'Senate': 0,
    'House committee actions': 1,
    'House floor actions': 2,
    'Library of Congress': 9
}

for i, bill_actions_url in enumerate(bill_details.loc[:, 'actionURL']):
    actions_request = requests.get(
        bill_actions_url,
        params={'api_key':secrets['Congress']['headers']['Authorization']}
    ).json()

    bill_info = actions_request['request'] # Acquire bill identifier
    bill_actions = actions_request['actions'] # Actions content

    bill_actions_source_list[i] = copy.deepcopy(bill_actions) # Not passing by reference
    
    for bill_action in bill_actions:
        try:
            # Just store the codes to join later
            bill_action['committeeCodes'] = [committee['systemCode'] for committee in bill_action['committees']]
            del bill_action['committees']
        except KeyError: # No committee when introduced from Library of Congress
            bill_action['committeeCodes'] = None

        try:
            # https://github.com/LibraryOfCongress/api.congress.gov/blob/main/Documentation/BillEndpoint.md#actions-level
            # sourceSystem can only have 4 values, which are outlined at link above
            bill_action['sourceSystemCode'] = bill_action['sourceSystem']['code']
            del bill_action['sourceSystem']
        except KeyError: # Some action sourceSystems don't contain a code key-value pair
            sourceSystemName = bill_action['sourceSystem']['name']
            bill_action['sourceSystemCode'] = sourceSystemMapper[sourceSystemName]

        bill_action['congress'] = bill_info['congress']
        bill_action['billNumber'] = bill_info['billNumber']
        bill_action['billType'] = bill_info['billType']

        bill_actions_list.append(bill_action)
    # print(json.dumps(bill_actions, indent=4))

In [116]:
with open(congress_input_dir / 'congress_bill_actions_sample.json', mode='w') as f:
    f.write(json.dumps(bill_actions_source_list, indent=4))

In [117]:
bill_actions = pd.DataFrame(bill_actions_list)
num_sample_bill_actions = len(bill_actions)

bill_actions.head()

,actionCode,actionDate,text,type,committeeCodes,sourceSystemCode,congress,billNumber,billType,sourceSystem,actionTime,recordedVotes,calendarNumber
0,H11000,2003-02-21,Referred to the Subcommittee on Education Reform.,Committee,[hsed14],1,108,5,hconres,NaN,NaN,NaN,NaN
1,H11100,2003-01-07,Referred to the House Committee on Education a...,IntroReferral,[hsed00],2,108,5,hconres,NaN,NaN,NaN,NaN
2,Intro-H,2003-01-07,Introduced in House,IntroReferral,None,9,108,5,hconres,NaN,NaN,NaN,NaN
3,1000,2003-01-07,Introduced in House,IntroReferral,None,9,108,5,hconres,NaN,NaN,NaN,NaN
4,H11000,2003-03-06,Referred to the Subcommittee on the Constitution.,Committee,[hsju10],1,108,32,hres,NaN,NaN,NaN,NaN


In [118]:
bill_actions.to_csv(congress_output_dir / 'congress_bill_actions_sample.csv')